# Phase 35 v1.0d — Grad-CAM Target-Mask Rank-Safety Amendment
## Reviewer Notebook (read-only)

**Classification:** `READ_ONLY_PHASE35_V1_0D_GRADCAM_RANK_REPAIR_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase35_interpretability_v1_0d`

Repairs the prospectively disclosed defect
`PHASE35_V1_0B_GRADCAM_TARGET_MASK_RANK_INCOMPATIBILITY` **before** the synthetic smoke re-run and
**before** any Phase35 scientific execution. Implementation-only amendment: no scientific protocol
change. **No scientific output exists and none is produced here.**


## 0. Setup

In [1]:
import hashlib
import json
from pathlib import Path
ROOT = Path(r"D://Softfors//jup"); AUD = ROOT / "reports" / "audits"
def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))
print("Phase35 v1.0d Grad-CAM target-mask rank-safety amendment review (READ-ONLY)")
print("No model construction, no checkpoint deserialization, no forward, no model autograd,")
print("no Grad-CAM execution, no router attribution, no images, no real p_global, no gate set.")


Phase35 v1.0d Grad-CAM target-mask rank-safety amendment review (READ-ONLY)
No model construction, no checkpoint deserialization, no forward, no model autograd,
no Grad-CAM execution, no router attribution, no images, no real p_global, no gate set.


## 1. Defect B — what was wrong

In [2]:
d = jload(AUD / "phase35_gradcam_target_rank_defect_v1_0d.json")
print("defect id   :", d["defect_id"])
print("classification:", d["classification"])
print("frozen defective component:", d["frozen_defective_component"])
print()
for k, v in d["known_scientific_tensors"].items():
    print(f"  {k:34s} {v}")
print()
print("problem:", d["problem"])
print()
print("flags:")
for k in ("scientific_specification_changed", "scientific_population_changed",
          "target_threshold_changed", "target_selection_source_changed",
          "causal_or_scientific_interpretation_changed", "implementation_rank_contract_changed",
          "scientific_result_invalidated", "frozen_v1_0b_modified", "frozen_v1_0c_modified"):
    print(f"  {k:46s} = {d[k]}")


defect id   : PHASE35_V1_0B_GRADCAM_TARGET_MASK_RANK_INCOMPATIBILITY
classification: PRE_SCIENTIFIC_GRADCAM_TARGET_IMPLEMENTATION_DEFECT__FROZEN_SCIENTIFIC_INTENT_UNCHANGED__NO_SCIENTIFIC_GRADCAM_EXECUTED
frozen defective component: phase35_interpretability_execution_v1_0b.gradcam_target_scalar (preserved byte-for-byte)

  canonical_G_logits                 {'rank': 4, 'shape': [1, 1, 512, 512], 'authority': 'GlobalMedFiLM.forward (out_channels=1)'}
  persisted_p_global                 {'rank': 2, 'shape': [512, 512], 'dtype': 'float16', 'authority': 'reports/predictions/phase21_global_medfilm_fourdomain_seed42_clean_v3/val/<sid>.npz[prob]'}
  build_target_mask_p_global         {'rank': 2, 'shape': [512, 512], 'dtype': 'Boolean', 'definition': 'persisted p_global >= 0.50'}

problem: The frozen implementation performs `logits[mask]`, attempting rank-2 Boolean indexing directly against rank-4 logits; PyTorch requires the mask shape to match the indexed tensor's leading dimensions, so thi

## 2. Frozen specification target intent (unchanged)

In [3]:
for k, v in d["frozen_scientific_intent"].items():
    print(f"  {k:22s} : {v}")
print()
print("evidence:")
for e in d["evidence"]:
    print("  -", e)


  M_pred                 : persisted p_global >= 0.50
  T_if_mask_nonempty     : mean(current_forward_foreground_logit[M_pred])
  T_if_mask_empty        : mean(current_forward_foreground_logit over full field)
  unchanged              : True

evidence:
  - reports/audits/phase35_v1_0b_smoke_stub_defect_record_v1_0c.json (defect_B)
  - model-free CPU tensor-semantics probe: x(1,1,4,4)[mask(4,4)] -> IndexError; x(1,1,4,4)[mask(1,1,4,4)] -> ok
  - metadata-only npy header inspection of all 808 VAL npz: prob = (512,512) '<f2', pred/gt = (512,512) '|u1' (the `gt` member was never accessed)
  - reproduced in this revision: unit test D1_frozen_v1_0b_logits_mask_indexing_reproduced -> IndexError


## 3. Rank-mismatch proof

In [4]:
import torch
logits = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
m2 = torch.zeros(4, 4, dtype=torch.bool); m2[0:2, 0:2] = True
print("canonical G logits rank/shape :", logits.ndim, tuple(logits.shape))
print("canonical scientific mask rank:", m2.ndim, tuple(m2.shape), m2.dtype)
try:
    _ = logits[m2]
    print("logits[mask] -> (unexpected) no error")
except Exception as e:
    print("frozen defective `logits[mask]` ->", type(e).__name__, ":", str(e)[:90])
print()
u = jload(AUD / "phase35_gradcam_target_rank_unit_v1_0d.json")
for r in u["results"]:
    if r["test"].startswith("D1"):
        print("reproduced in the frozen unit artifact:", r["test"], "->", r["observed"])


canonical G logits rank/shape : 4 (1, 1, 4, 4)
canonical scientific mask rank: 2 (4, 4) torch.bool
frozen defective `logits[mask]` -> IndexError : The shape of the mask [4, 4] at index 0 does not match the shape of the indexed tensor [1,

reproduced in the frozen unit artifact: D1_frozen_v1_0b_logits_mask_indexing_reproduced -> {'raised': True, 'exc': 'IndexError'}


## 4. Exact rank-safe contract (frozen by v1.0d)

In [5]:
t = jload(AUD / "phase35_gradcam_rank_repair_terminal_v1_0d.json")
for k, v in t["rank_safe_contract"].items():
    print(f"  {k:14s} = {v}")
print()
import re
src = (ROOT / "scripts/analysis/phase35_interpretability_execution_v1_0d.py").read_text(encoding="utf-8")
body = src.split("def gradcam_target_scalar_rank_safe(logits, mask):")[1].split("def gradcam_gradient")[0]
for line in body.splitlines():
    if line.strip() and not line.strip().startswith("#"):
        print(line)


  logits_rank    = 4
  logits_shape   = [1,1,H,W]
  batch          = 1
  channels       = 1
  mask_rank      = 2
  mask_shape     = [H,W]
  mask_dtype     = bool

    """Rank-safe equivalent of the frozen scientific target scalar.
    Frozen intent (unchanged):
      M_pred = persisted p_global >= 0.50
      if M_pred non-empty:  T = mean(foreground logits under M_pred)
      else:                 T = mean(foreground logits over the full field)
    The rank-2 mask is the canonical representation. It is NEVER broadcast, never expanded to
    unrelated batch/channel dimensions, never created from the current forward, never the ROI-0.30
    mask, never GT, and never a sigmoid probability target.
    Implemented with explicit raises rather than `assert` so the fail-closed property survives
    `python -O`; semantics are identical to the specified assert-based contract.
    """
    logits_f = _validate_target_logits(logits)
    mask_t = _validate_target_mask(logits_f, mask)
    foreground_l

## 5. Model-free unit-test results (CPU only)

In [6]:
print("tested object:", u["tested_object"]["function"], "in", u["tested_object"]["module"])
print("device:", u["device"], "| cuda used:", u["cuda_used"])
print()
for r in u["results"]:
    print(f"  [{'PASS' if r['pass'] else 'FAIL'}] {r['test']:52s} expected={r['expected']} observed={r['observed']}")
print()
print("TOTAL:", u["n_tests"], "tests,", u["n_failed"], "failed | all_pass:", u["all_pass"])
print()
print("execution counters:", json.dumps(u["execution_counters"], indent=1))


tested object: gradcam_target_scalar_rank_safe in scripts/analysis/phase35_interpretability_execution_v1_0d.py
device: cpu | cuda used: False

  [PASS] T1_nonempty_rank2_mask                               expected=2.5 observed=2.5
  [PASS] T2_empty_mask_full_field_fallback                    expected=7.5 observed=7.5
  [PASS] T3_spatial_mismatch_rejected                         expected=fail_closed observed={'raised': True, 'exc': 'RuntimeError'}
  [PASS] T4_rank4_mask_rejected                               expected=fail_closed observed={'raised': True, 'exc': 'RuntimeError'}
  [PASS] T5a_float_mask_rejected                              expected=fail_closed observed={'raised': True, 'exc': 'RuntimeError'}
  [PASS] T5b_integer_mask_rejected                            expected=fail_closed observed={'raised': True, 'exc': 'RuntimeError'}
  [PASS] T6_logits_rank3_rejected                             expected=fail_closed observed={'raised': True, 'exc': 'RuntimeError'}
  [PASS] T7_logits_ba

## 6. Semantic diff v1.0c → v1.0d

In [7]:
df = jload(AUD / "phase35_v1_0c_to_v1_0d_semantic_diff.json")
for k in ("scientific_specification_changed", "gradcam_scalar_scientific_intent_changed",
          "target_population_changed", "target_mask_source_changed", "target_threshold_changed",
          "empty_mask_fallback_changed", "gradcam_layers_changed", "gradcam_algorithm_changed",
          "cam_normalization_changed", "precision_context_changed", "router_science_changed",
          "film_science_changed"):
    print(f"  {k:46s} = {df[k]}")
print()
print("only implementation change :", df["only_implementation_change"])
print("scientific change class    :", df["scientific_semantic_change_class"])
print("functions added in v1.0d   :", df["functions_added_in_v1_0d"])
print("functions wrapped in v1.0d :", df["functions_wrapped_in_v1_0d"])
print("thresholds                 :", df["thresholds"])


  scientific_specification_changed               = False
  gradcam_scalar_scientific_intent_changed       = False
  target_population_changed                      = False
  target_mask_source_changed                     = False
  target_threshold_changed                       = False
  empty_mask_fallback_changed                    = False
  gradcam_layers_changed                         = False
  gradcam_algorithm_changed                      = False
  cam_normalization_changed                      = False
  precision_context_changed                      = False
  router_science_changed                         = False
  film_science_changed                           = False

only implementation change : TARGET_MASK_RANK_SAFE_INDEXING
scientific change class    : IMPLEMENTATION_SHAPE_SAFETY_FIX_ONLY
functions added in v1.0d   : ['_hook', '_validate_target_logits', '_validate_target_mask', 'gradcam_gradient_enabled_rank_safe', 'gradcam_path_contract', 'gradcam_target_scalar_rank_safe', 

## 7. Call graph — modes reach the repaired helper

In [8]:
pf = jload(AUD / "phase35_execution_source_static_preflight_v1_0d.json")
for k, v in pf["call_graph_reachability"].items():
    print(f"  {k:66s} = {v}")


  mode_gradcam_to_gradcam_target_scalar_rank_safe                    = True
  mode_gradcam_to_gradcam_gradient_enabled_rank_safe                 = True
  mode_science_to_gradcam_target_scalar_rank_safe                    = True
  mode_smoke_to_smoke_gradcam                                        = True
  smoke_gradcam_to_gradcam_target_scalar_rank_safe                   = True
  gradcam_gradient_enabled_rank_safe_to_frozen_defective_target_scalar = False


## 8. Static preflight (zero execution)

In [9]:
print("effective execution source set:", pf["effective_execution_source_set"])
print("predecessors preserved:", pf["n_predecessors_preserved"],
      "| all match:", pf["preservation_all_match"])
print("original 54 checks :", pf["original_n_checks"], "| failed:", pf["original_n_failed"])
print("additional v1.0d   :", pf["additional_n_checks"], "| failed:", pf["additional_n_failed"])
print("rank-safety checks :", pf["rank_safety_n_checks"], "| failed:", pf["rank_safety_n_failed"])
print("TOTAL              :", pf["total_n_checks"], "checks,", pf["total_n_failed"], "failed")
print("static_source_preflight_pass:", pf["static_source_preflight_pass"])
print()
print("rank-safety checks:")
for k, v in pf["rank_safety_checks"].items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")


effective execution source set: ['scripts/analysis/phase35_interpretability_execution_v1_0d.py', 'scripts/analysis/phase35_interpretability_execution_v1_0b.py']
predecessors preserved: 33 | all match: True
original 54 checks : 54 | failed: 0
additional v1.0d   : 22 | failed: 0
rank-safety checks : 20 | failed: 0
TOTAL              : 96 checks, 0 failed
static_source_preflight_pass: True

rank-safety checks:
  [PASS] 01_defective_v1_0b_logits_mask_scientific_path_unreachable
  [PASS] 02_target_helper_requires_logits_rank_4
  [PASS] 03_batch_dimension_exactly_1
  [PASS] 04_output_channel_dimension_exactly_1
  [PASS] 05_target_mask_rank_exactly_2
  [PASS] 06_spatial_dimensions_exact_match
  [PASS] 07_mask_boolean
  [PASS] 08_target_plane_explicitly_logits_0_0
  [PASS] 09_nonempty_target_selects_only_mask_pixels
  [PASS] 10_empty_mask_fallback_full_plane_mean
  [PASS] 11_persisted_p_global_still_sole_scientific_mask_authority
  [PASS] 12_segmentation_threshold_still_0p50
  [PASS] 13_ROI_th

## 9. Source freeze

In [10]:
fz = jload(AUD / "phase35_preexecution_source_freeze_v1_0d.json")
for name, spec in fz["frozen_source_set"].items():
    print(f"  {name:32s} {spec['sha256']}")
print()
for k in ("v1_0_spec_preserved", "v1_0b_source_preserved", "v1_0c_source_preserved",
          "v1_0c_smoke_amendment_preserved", "v1_0d_rank_repair_complete",
          "future_smoke_must_use_exact_v1_0d_bytes", "first_scientific_execution_started"):
    print(f"  {k:44s} = {fz[k]}")
print()
print("frozen authorities:")
for name, spec in fz["frozen_authorities"].items():
    print(f"  {name:38s} {spec.get('sha256','')}")


  execution_v1_0d                  cccc109ade27eb5ad7883912004c1e911e9e253a659146f3d64820af92244056
  static_preflight_v1_0d           07cdab4b4eeb2051c2007ff5bdfaf3dba8b91c6e510b4a11979c0def53bbf356
  rank_unit_test_v1_0d             6c3896b01e60c5e8361d0af9d98ebd29c5b592a906257c487c6c9140242ec19f
  rank_unit_results_v1_0d          66a16c411b30852e32701f424384b925b42ab074bf2c199521f422f5b5e58933
  static_preflight_v1_0d_result    6d1438e139436cef7c08c9eca52d5a4f068f1412c4e45f22c60cdb4d90783cb3
  common_v1_0b_reused              9963215e1431e968e11bacbcd917d96a8a985a7181000258b90f458ff0a778ac
  execution_v1_0c_authority        4058fc5544e6e1af15339df20a88da4880549ea8d4dd434395df04e0fa261baa
  execution_v1_0b_reused           581926840d7717e19ed15ad77c7c5339b709efc881b938924d6eec0a2ceb1872

  v1_0_spec_preserved                          = True
  v1_0b_source_preserved                       = True
  v1_0c_source_preserved                       = True
  v1_0c_smoke_amendment_preserved    

## 10. Eligibility adjudication

In [11]:
el = jload(AUD / "phase35_execution_eligibility_adjudication_v1_0d.json")
for k in ("film_parameter_analysis_source_eligible", "router_VAL808_attribution_source_eligible",
          "gradcam_bridge_source_eligible", "gradcam_decoder1_source_eligible",
          "first_scientific_gradcam_execution_eligible",
          "first_phase35_scientific_execution_eligible",
          "scientific_gradcam_authorized_by_static_checks_alone",
          "synthetic_smoke_reexecution_eligible", "synthetic_smoke_executed_in_v1_0d"):
    print(f"  {k:56s} = {el[k]}")
print()
print("block reason:", el["first_scientific_gradcam_execution_block_reason"])
print("gates:", el["smoke_gate_env"], "|", el["science_gate_env"])


  film_parameter_analysis_source_eligible                  = True
  router_VAL808_attribution_source_eligible                = True
  gradcam_bridge_source_eligible                           = True
  gradcam_decoder1_source_eligible                         = True
  first_scientific_gradcam_execution_eligible              = False
  first_phase35_scientific_execution_eligible              = False
  scientific_gradcam_authorized_by_static_checks_alone     = False
  synthetic_smoke_reexecution_eligible                     = True
  synthetic_smoke_executed_in_v1_0d                        = False

block reason: PENDING_SUCCESSFUL_V1_0D_SYNTHETIC_SMOKE
gates: PHASE35_SMOKE_EXECUTE | PHASE35_EXECUTE


## 11. No-model-execution audit

In [12]:
ne = jload(AUD / "phase35_v1_0d_no_model_execution_audit.json")
for k, v in ne["model_execution_counters"].items():
    print(f"  {k:34s} = {v}")
print()
print("allowed separately (recorded distinctly):")
for k, v in ne["allowed_separately"].items():
    print(f"  {k:44s} = {v}")


  model_constructions                = 0
  checkpoint_deserializations        = 0
  strict_loads                       = 0
  forwards                           = 0
  autograd_calls                     = 0
  gradcam_executions                 = 0
  router_attribution_executions      = 0
  raw_val_reads                      = 0
  real_persisted_p_global_reads      = 0
  gt_reads                           = 0
  router_holdout_reads               = 0
  test_reads                         = 0
  training                           = 0
  gate_env_vars_set                  = 0
  mode_invocations                   = 0

allowed separately (recorded distinctly):
  model_free_cpu_rank_unit_tests               = 11
  model_free_cpu_rank_unit_tests_failed        = 0
  model_free_cpu_rank_unit_tests_artifact      = {'path': 'reports/audits/phase35_gradcam_target_rank_unit_v1_0d.json', 'sha256': '66a16c411b30852e32701f424384b925b42ab074bf2c199521f422f5b5e58933'}
  device                                 

## 12. Final check and terminal

In [13]:
fc = jload(AUD / "phase35_gradcam_rank_repair_final_check_v1_0d.json")
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
print("defect:", fc["defect"], "->", fc["defect_status"])
print()
tm = jload(AUD / "phase35_gradcam_rank_repair_terminal_v1_0d.json")
print("terminal:"); print(" ", tm["terminal"])


final check: 46 checks, 0 failed
defect: PHASE35_V1_0B_GRADCAM_TARGET_MASK_RANK_INCOMPATIBILITY -> REPAIRED_IN_V1_0D

terminal:
  PHASE35_PREEXECUTION_GRADCAM_RANK_REPAIR_V1_0D_COMPLETE__DISCLOSED_V1_0B_TARGET_MASK_RANK_INCOMPATIBILITY_REPAIRED_WITH_EXPLICIT_BATCH1_CHANNEL1_RANK2_MASK_CONTRACT__PERSISTED_PGLOBAL050_TARGET_AND_EMPTY_MASK_FALLBACK_PRESERVED__STANDALONE_AND_SCIENCE_GRADCAM_PATHS_BOUND_TO_RANK_SAFE_TARGET__V1_0D_SMOKE_BOUND_TO_SAME_SCIENTIFIC_TARGET_HELPER__NO_SCIENTIFIC_PROTOCOL_CHANGE__V1_0D_SOURCE_FREEZE_COMPLETE__SYNTHETIC_SMOKE_ELIGIBLE_NOT_STARTED


---
## Summary

| item | value |
|---|---|
| defect `PHASE35_V1_0B_GRADCAM_TARGET_MASK_RANK_INCOMPATIBILITY` | **REPAIRED** (implementation only) |
| scientific specification / target source / threshold / fallback | **UNCHANGED** |
| canonical logits contract | rank 4, `[1,1,H,W]`, batch 1, channels 1 |
| canonical mask contract | rank 2, `[H,W]`, Boolean |
| repaired helper | `gradcam_target_scalar_rank_safe` |
| `mode_gradcam` / `mode_science` / `mode_smoke` → repaired helper | YES / YES / YES |
| frozen defective `logits[mask]` path | UNREACHABLE |
| static preflight | 96 checks · 0 failed · 34 predecessors byte-identical |
| model-free CPU rank unit tests | 11 tests · 0 failed |
| v1.0d source freeze | COMPLETE |
| synthetic smoke | ELIGIBLE, NOT STARTED |
| first scientific Grad-CAM execution | NOT eligible (pending successful v1.0d smoke) |
| model constructions · checkpoint loads · forwards · model autograd | 0 · 0 · 0 · 0 |
| raw VAL · real p_global · GT · ROUTER_HOLDOUT · TEST | 0 · 0 · 0 · 0 · 0 |
| Phase35 scientific execution | NOT started · Phase36 NOT started · TEST locked |
